# HRSC2016 + Strip R-CNN-S Notebook

This notebook is tailored to the HRSC2016-trained Strip R-CNN-S checkpoint in this repository. It gives you three workflows:

- load a pretrained Strip R-CNN-S detector
- render detections for only the classes you choose
- evaluate either the full annotated split or a selected subset of images

The notebook assumes the repository was installed with `pip install -e .` and that your HRSC2016 data follows the layout expected by the config.

In [ ]:
from __future__ import annotations

import copy
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'configs').exists() and (cache_path := candidate / 'setup.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the Strip-R-CNN repository root from the current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import mmcv
import numpy as np
import torch
from mmcv import Config
from mmcv.cnn.utils import revert_sync_batchnorm
from mmcv.parallel import MMDataParallel, scatter
from mmdet.apis import init_detector, inference_detector, single_gpu_test
from mmdet.datasets import build_dataloader

import mmrotate  # noqa: F401
from mmrotate.apis import inference_detector_by_patches
from mmrotate.datasets import build_dataset

print(f'Repository root: {REPO_ROOT}')
print(f'Torch version: {torch.__version__}')
print(f'MMRotate version: {mmrotate.__version__}')



In [ ]:


# Since HRSC classes are often just one or a few, we define them here for convenience,
# but always check model.CLASSES after loading the model.

CONFIG_PATH = REPO_ROOT / 'configs/strip_rcnn/strip_regex_s_fpn_3x_hrsc_le90.py'
CHECKPOINT_PATH = REPO_ROOT / 'weights/strip_rcnn_s_dota.pth'
# Note: You should use an HRSC-trained checkpoint if available.
# The above is a placeholder from the DOTA notebook.

HRSC_ROOT = REPO_ROOT / 'mmrotate/data/HRSC2016'
TEST_IMAGE_DIR = HRSC_ROOT / 'FullDataSet/AllImages'

CUDA_DEVICE_NAME = None
CUDA_CAPABILITY = None
SUPPORTED_CUDA_ARCHES = []
if torch.cuda.is_available():
    CUDA_DEVICE_NAME = torch.cuda.get_device_name(0)
    CUDA_CAPABILITY = torch.cuda.get_device_capability(0)
    SUPPORTED_CUDA_ARCHES = sorted(torch.cuda.get_arch_list())


def pick_device() -> str:
    if not torch.cuda.is_available():
        return 'cpu'

    current_arch = f'sm_{CUDA_CAPABILITY[0]}{CUDA_CAPABILITY[1]}'
    if current_arch not in SUPPORTED_CUDA_ARCHES:
        print(
            f'CUDA device {CUDA_DEVICE_NAME} is visible, but this torch build does not support {current_arch}. '
            f'Supported arches: {SUPPORTED_CUDA_ARCHES}. Falling back to CPU. Upgrade PyTorch/CUDA to use this GPU.'
        )
        return 'cpu'

    return 'cuda:0'


DEVICE = pick_device()

RENDER_CLASSES = ['ship'] # Placeholder
VIS_SCORE_THR = 0.30
USE_PATCH_INFERENCE = False
PATCH_SIZES = [1024]
PATCH_STEPS = [824]
IMG_RATIOS = [1.0]
MERGE_IOU_THR = 0.10

TEST_IMAGE_CANDIDATES = sorted(TEST_IMAGE_DIR.glob('*.png'))
if not TEST_IMAGE_CANDIDATES:
    raise FileNotFoundError(f'No PNG images found under {TEST_IMAGE_DIR}')
SAMPLE_IMAGE = TEST_IMAGE_CANDIDATES[0]
MAX_VIS_IMAGES = 8

EVAL_SPLIT = 'test'
METRIC = None
SAMPLES_PER_GPU = 1
WORKERS_PER_GPU = 2

SELECTED_IMAGE_IDS = []
MAX_SUBSET_IMAGES = 8 if DEVICE == 'cpu' else None

print('Config path:', CONFIG_PATH)
print('Checkpoint path:', CHECKPOINT_PATH)
print('HRSC root:', HRSC_ROOT)
print('Test image dir:', TEST_IMAGE_DIR)
print('CUDA device:', CUDA_DEVICE_NAME)
print('CUDA capability:', CUDA_CAPABILITY)
print('Supported torch CUDA arches:', SUPPORTED_CUDA_ARCHES)
print('Device:', DEVICE)
print('Default render classes:', RENDER_CLASSES)
print('Sample image:', SAMPLE_IMAGE.name)
print('Test images found:', len(TEST_IMAGE_CANDIDATES))
print('Default subset size:', MAX_SUBSET_IMAGES)



In [ ]:

def validate_render_classes(selected_classes, all_classes):
    selected = list(selected_classes or all_classes)
    unknown = [class_name for class_name in selected if class_name not in all_classes]
    if unknown:
        raise ValueError(f'Unknown class names: {unknown}. Valid HRSC classes are: {all_classes}')
    return selected


def configure_hrsc_paths(cfg: Config, dataset_root: Path) -> Config:
    cfg = copy.deepcopy(cfg)
    dataset_root = Path(dataset_root)

    if cfg.model.get('backbone') and cfg.model.backbone.get('init_cfg'):
        cfg.model.backbone.init_cfg = None
    if cfg.model.get('pretrained', None) is not None:
        cfg.model.pretrained = None

    # HRSC layout from hrsc.py
    cfg.data.train.ann_file = str(dataset_root / 'ImageSets/trainval.txt')
    cfg.data.train.img_prefix = str(dataset_root / 'FullDataSet/AllImages/')
    cfg.data.train.ann_subdir = str(dataset_root / 'FullSD/Annotations/') # This might be wrong based on reading config

    cfg.data.val.ann_file = str(dataset_root / 'ImageSets/test.txt')
    cfg.data.val.img_prefix = str(dataset_root / 'FullDataSet/AllImages/')
    cfg.data.val.ann_subdir = str(dataset_root / 'FullSD/Annotations/')

    cfg.data.test.ann_file = str(dataset_root / 'ImageSets/test.images/') # This is also likely wrong

    cfg.data.test.img_prefix = str(dataset_root / 'FullDataSet/AllImages/')
    cfg.data.test.img_prefix = str(dataset_root / 'FullDataSet/AllImages/')

    return cfg


def load_strip_rcnn_model(config_path: Path, checkpoint_path: Path, dataset_root: Path, device: str = DEVICE):
    config_path = Path(config_path)
    checkpoint_path = Path(checkpoint_path)
    dataset_root = Path(dataset_root)

    if not config_path.exists():
        raise FileNotFoundError(f'Config file not found: {config_path}')
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f'Checkpoint not found: {checkpoint_path}. Download the full DOTA detector checkpoint and update CHECKPOINT_PATH.'
        )
    if not dataset_root.exists():
        raise FileNotFoundError(f'Dataset root not found: {dataset_root}')